In [20]:
import tensorflow as tf

from matplotlib import pyplot as plt
from PIL import Image
import numpy as np
import os
from os import listdir


import numpy as np
import os
import PIL
import PIL.Image
import tensorflow as tf


In [21]:

import os
import numpy as np
import tensorflow as tf

IMAGE_SIZE = 224
BATCH_SIZE = 8
SEED = 42

folders = [
    ("Big_Dataset/not_robin", 0),
    ("Big_Dataset/robin", 1),
]

all_paths = []
all_labels = []

for folder, label in folders:
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")

    for filename in os.listdir(folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths.append(os.path.join(folder, filename))
            all_labels.append(label)

all_paths = np.asarray(all_paths)
all_labels = np.asarray(all_labels, dtype=np.int32)

if len(all_paths) == 0:
    raise ValueError("No supported images found.")

print("Total images:", len(all_paths))
print("Not Robin:", np.sum(all_labels == 0))
print("Robin:", np.sum(all_labels == 1))

# Split each class separately so both sets contain both classes
rng = np.random.default_rng(SEED)
train_indices = []
val_indices = []

for class_label in (0, 1):
    class_indices = np.where(all_labels == class_label)[0]
    rng.shuffle(class_indices)

    if len(class_indices) < 2:
        raise ValueError(
            f"Need at least 2 images for class {class_label}."
        )

    n_val = max(1, int(len(class_indices) * 0.2))
    n_val = min(n_val, len(class_indices) - 1)

    val_indices.extend(class_indices[:n_val])
    train_indices.extend(class_indices[n_val:])

rng.shuffle(train_indices)
rng.shuffle(val_indices)

train_paths = all_paths[train_indices]
train_labels = all_labels[train_indices]

val_paths = all_paths[val_indices]
val_labels = all_labels[val_indices]

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))




Total images: 1952
Not Robin: 1095
Robin: 857
Training images: 1562
Validation images: 390


In [22]:

def load_image(path, label):
    image_bytes = tf.io.read_file(path)

    image = tf.io.decode_image(
        image_bytes,
        channels=1,
        expand_animations=False
    )

    image.set_shape([None, None, 1])

    image = tf.image.resize(
        image, [224, 224]
    )
    image = tf.cast(image, tf.float32) / 255.0

    label = tf.reshape(
        tf.cast(label, tf.float32), [1]
    )

    return image, label


In [23]:

import tensorflow as tf
import numpy as np

learning_rate = 0.001

W1 = tf.Variable(tf.random.normal([3, 3, 1, 8], stddev=0.1))
b1 = tf.Variable(tf.zeros([8]))

W2 = tf.Variable(tf.random.normal([3, 3, 8, 16], stddev=0.1))
b2 = tf.Variable(tf.zeros([16]))

# Global average pooling produces 16 features
W3 = tf.Variable(tf.random.normal([16, 64], stddev=0.01))
b3 = tf.Variable(tf.zeros([64]))

W4 = tf.Variable(tf.random.normal([64, 1], stddev=0.01))
b4 = tf.Variable(tf.zeros([1]))

trainable_variables = [W1, b1, W2, b2, W3, b3, W4, b4]


def forward(x):
    x = tf.nn.conv2d(x, W1, strides=1, padding="SAME") + b1
    x = tf.nn.relu(x)
    x = tf.nn.max_pool2d(x, ksize=2, strides=2, padding="VALID")

    x = tf.nn.conv2d(x, W2, strides=1, padding="SAME") + b2
    x = tf.nn.relu(x)
    x = tf.nn.max_pool2d(x, ksize=2, strides=2, padding="VALID")

    # Average over the height and width dimensions
    x = tf.reduce_mean(x, axis=[1, 2])

    x = tf.nn.relu(tf.matmul(x, W3) + b3)

    logits = tf.matmul(x, W4) + b4
    return logits


optimizer = tf.optimizers.Adam(learning_rate=learning_rate)


def train_step(images, labels):
    with tf.GradientTape() as tape:
        logits = forward(images)

        labels = tf.cast(labels, tf.float32)
        labels = tf.reshape(labels, tf.shape(logits))

        loss_value = tf.reduce_mean(
            tf.nn.sigmoid_cross_entropy_with_logits(
                labels=labels,
                logits=logits
            )
        )

    gradients = tape.gradient(
        loss_value, trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, trainable_variables)
    )

    return loss_value

In [24]:
def loss(y_true, y_pred):

    eps = 1e-7

    return -(
        y_true*np.log(y_pred+eps)
        +
        (1-y_true)*np.log(
            1-y_pred+eps
        )
    )

In [25]:
train_ds = make_dataset(
    train_paths, train_labels, training=True
)

val_ds = make_dataset(
    val_paths, val_labels, training=False
)

In [26]:
for images, labels in train_ds.take(1):
    logits = forward(images)

    print("Images:", images.shape)
    print("Labels:", labels.shape)
    print("Logits:", logits.shape)

Images: (8, 224, 224, 1)
Labels: (8, 1)
Logits: (8, 1)


In [27]:
epochs = 10

for images, labels in train_ds.take(1):
    print("Images:", images.shape)  # (8, 224, 224, 1)
    print("Labels:", labels.shape)  # (8, 1)

for epoch in range(epochs):
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for images, labels in train_ds:
        loss_value = train_step(images, labels)

        logits = forward(images)
        predictions = tf.cast(logits >= 0.0, tf.float32)

        batch_size = int(tf.shape(labels)[0])

        total_loss += float(loss_value) * batch_size
        total_correct += int(
            tf.reduce_sum(
                tf.cast(predictions == labels, tf.int32)
            )
        )
        total_seen += batch_size

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"Loss={total_loss / total_seen:.4f} "
        f"Accuracy={total_correct / total_seen:.2%}"
    )

Images: (8, 224, 224, 1)
Labels: (8, 1)


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} channels must be 0 or 3 for GIF, got 1
	 [[{{node decode_image/DecodeImage}}]] [Op:IteratorGetNext] name: 

In [28]:

for batch_number, (images, labels) in enumerate(train_ds):
    print(
        f"Batch {batch_number}: "
        f"images={images.shape}, labels={labels.shape}"
    )

print("All batches loaded successfully.")


Batch 0: images=(8, 224, 224, 1), labels=(8, 1)
Batch 1: images=(8, 224, 224, 1), labels=(8, 1)
Batch 2: images=(8, 224, 224, 1), labels=(8, 1)
Batch 3: images=(8, 224, 224, 1), labels=(8, 1)
Batch 4: images=(8, 224, 224, 1), labels=(8, 1)
Batch 5: images=(8, 224, 224, 1), labels=(8, 1)
Batch 6: images=(8, 224, 224, 1), labels=(8, 1)
Batch 7: images=(8, 224, 224, 1), labels=(8, 1)
Batch 8: images=(8, 224, 224, 1), labels=(8, 1)
Batch 9: images=(8, 224, 224, 1), labels=(8, 1)
Batch 10: images=(8, 224, 224, 1), labels=(8, 1)
Batch 11: images=(8, 224, 224, 1), labels=(8, 1)
Batch 12: images=(8, 224, 224, 1), labels=(8, 1)
Batch 13: images=(8, 224, 224, 1), labels=(8, 1)
Batch 14: images=(8, 224, 224, 1), labels=(8, 1)
Batch 15: images=(8, 224, 224, 1), labels=(8, 1)
Batch 16: images=(8, 224, 224, 1), labels=(8, 1)
Batch 17: images=(8, 224, 224, 1), labels=(8, 1)
Batch 18: images=(8, 224, 224, 1), labels=(8, 1)
Batch 19: images=(8, 224, 224, 1), labels=(8, 1)
Batch 20: images=(8, 224, 224,

InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} channels must be 0 or 3 for GIF, got 1
	 [[{{node decode_image/DecodeImage}}]] [Op:IteratorGetNext] name: 